# Week 3 — SQL for Health Data: Connecting from Python

**Block 1, Essentials of Biomedical Informatics**

This week's SQL work lives in `Week3_Cohort_Query.sql` — start there. This notebook shows the other half of the picture: how to run that same SQL *from Python* and pull the result straight into a pandas DataFrame, so you can move seamlessly from database to analysis.

## Learning objectives
- Connect to a DuckDB database from Python
- Run a parameterized SQL query and load the result into pandas
- Understand why "push the filtering into SQL, then analyze in Python" is usually faster than pulling raw tables and filtering in pandas

## Part 1 — Walkthrough

### Step 1: Connect to the database

In [ ]:
import duckdb
import pandas as pd

con = duckdb.connect("synthetic_omop.duckdb")
con.execute("SHOW TABLES").df()

### Step 2: Run the cohort query from Week 3's SQL file

This is the exact same CTE + window function query from `Week3_Cohort_Query.sql`, Step 3 — just executed from Python instead of a SQL client.

In [ ]:
query = """
WITH t2d_dx AS (
    SELECT person_id, condition_start_date AS diagnosis_date
    FROM condition_occurrence
    WHERE condition_concept_id = 101
),
metformin_ranked AS (
    SELECT
        person_id,
        drug_exposure_start_date,
        ROW_NUMBER() OVER (PARTITION BY person_id ORDER BY drug_exposure_start_date) AS rn
    FROM drug_exposure
    WHERE drug_concept_id = 201
),
first_metformin AS (
    SELECT person_id, drug_exposure_start_date AS metformin_start
    FROM metformin_ranked
    WHERE rn = 1
)
SELECT
    p.person_id,
    g.concept_name AS sex,
    t.diagnosis_date,
    f.metformin_start,
    FLOOR((t.diagnosis_date - CAST((p.year_of_birth || '-01-01') AS DATE)) / 365.25) AS age_at_diagnosis
FROM t2d_dx t
JOIN first_metformin f ON f.person_id = t.person_id
JOIN person p ON p.person_id = t.person_id
LEFT JOIN concept g ON p.gender_concept_id = g.concept_id
WHERE f.metformin_start BETWEEN t.diagnosis_date AND t.diagnosis_date + INTERVAL 90 DAY
"""

cohort = con.execute(query).df()
cohort.head()

### Step 3: Now that it's a DataFrame, analyze it like any other pandas object

In [ ]:
print(cohort.shape)
cohort.groupby("sex")["age_at_diagnosis"].describe()

### Why filter in SQL instead of pandas?

You could have pulled the entire `condition_occurrence`, `drug_exposure`, and `person` tables into pandas and done the joins/filtering there. On this tiny teaching database it wouldn't matter. On a real OMOP database with hundreds of millions of rows, pulling everything into memory first is often impossible — the database engine is built to do joins and filters efficiently on disk. **General rule: push filtering and joining into SQL; bring back only the rows and columns you actually need.**

## Part 2 — Your Turn (this week's deliverable)

1. Adapt the query above to instead build the **hypertension → lisinopril within 180 days** cohort from `Week3_Cohort_Query.sql`'s Task 1/2 (condition_concept_id = 102, drug_concept_id = 202, 180-day window).
2. Load it into a DataFrame and report the mean age at diagnosis by sex, as in Step 3.
3. Export your cohort to a CSV file called `week3_cohort_hypertension.csv` using `.to_csv()`.
4. Submit this notebook (with your completed cells) via the Week 3 dropbox.

In [ ]:
# Task 1 & 2: adapt the query for the hypertension/lisinopril cohort


In [ ]:
# Task 3: export to CSV
